# 03 - Inference
Three ways to use the trained model:
1. load the joblib pipeline directly,
2. load the registered MLflow model (`models:/heart-disease-classifier@champion`),
3. call the REST API (`/predict`).

In [1]:
import os, sys, warnings
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from IPython.display import Image, display
from heart import config
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid")

In [2]:
import json, joblib
model = joblib.load(config.MODEL_PATH)
model

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(add_indicator=True,
                                                                                 strategy='median')),
                                                                  ('scale',
                                                                   StandardScaler())]),
                                                  ['age', 'trestbps', 'chol',
                                                   'thalach', 'oldpeak',
                                                   'ca']),
                                                 ('bin',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='most_frequent'))]),
                                                  ['sex', 'fbs', 'exang']),
                                                 ('cat',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(fill_value=-1,
                                                                                 strategy='constant')),
                                                                  ('onehot',
                                                                   OneHotEncoder(categories=[[-1,
                                                                                              1,
                                                                                              2,
                                                                                              3,
                                                                                              4],
                                                                                             [-1,
                                                                                              0,
                                                                                              1,
                                                                                              2],
                                                                                             [-1,
                                                                                              1,
                                                                                              2,
                                                                                              3],
                                                                                             [-1,
                                                                                              3,
                                                                                              6,
                                                                                              7]],
                                                                                 handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['cp', 'restecg', 'slope',
                                                   'thal'])])),
                ('model',
                 LogisticRegression(C=0.5, max_iter=2000, penalty='l1',
                                    random_state=42, solver='liblinear'))])

In [3]:
patients = [json.loads(Path(f"samples/{n}.json").read_text())
            for n in ["patient_high_risk", "patient_low_risk", "patient_partial"]]
frame = pd.DataFrame(patients).reindex(columns=config.FEATURES).astype(float)
frame.assign(p_disease=model.predict_proba(frame)[:, 1].round(4), prediction=model.predict(frame))

,age,trestbps,chol,thalach,oldpeak,ca,sex,fbs,exang,cp,restecg,slope,thal,p_disease,prediction
0,58.0,140.0,289.0,135.0,1.8,1.0,1.0,0.0,1.0,4.0,2.0,2.0,7.0,0.9624,1
1,41.0,126.0,306.0,163.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,1.0,3.0,0.0150,0
2,63.0,150.0,NaN,110.0,2.0,NaN,1.0,NaN,1.0,4.0,NaN,NaN,NaN,0.9471,1


Note the third patient has no `chol`, `ca`, `thal`, `slope` - the pipeline imputes them exactly as in training.

## MLflow model registry

In [4]:
import mlflow
mlflow.set_tracking_uri(f"sqlite:///{(config.ROOT / 'mlflow.db').as_posix()}")
registered = mlflow.sklearn.load_model("models:/heart-disease-classifier@champion")
np.allclose(registered.predict_proba(frame), model.predict_proba(frame))

True

## REST API (in-process test client - identical code path to the container)

In [5]:
from fastapi.testclient import TestClient
from heart.serving import app
with TestClient(app) as client:
    for p in patients:
        r = client.post("/predict", json=p)
        print(r.status_code, r.json())
    bad = client.post("/predict", json={"age": 58, "cp": 9})
    print(bad.status_code, bad.json()["detail"][0]["msg"])

{"ts": "2026-10-08T11:03:32-0400", "level": "INFO", "logger": "heart.api", "msg": "model loaded", "path": "C:\\Users\\karth\\heart-disease-mlops\\models\\heart_model.joblib", "model": "logistic_regression"}


{"ts": "2026-10-08T11:03:32-0400", "level": "INFO", "logger": "heart.api", "msg": "prediction", "prediction": 1, "probability": 0.9624}


{"ts": "2026-10-08T11:03:32-0400", "level": "INFO", "logger": "heart.api", "msg": "request", "request_id": "52b2b693daac", "method": "POST", "path": "/predict", "status": 200, "latency_ms": 47.3, "client": "testclient"}


200 {'prediction': 1, 'label': 'heart disease', 'confidence': 0.9624, 'probability_disease': 0.9624, 'risk_band': 'high', 'model_name': 'logistic_regression', 'model_version': '1.0.0'}
{"ts": "2026-10-08T11:03:32-0400", "level": "INFO", "logger": "heart.api", "msg": "prediction", "prediction": 0, "probability": 0.015}


{"ts": "2026-10-08T11:03:32-0400", "level": "INFO", "logger": "heart.api", "msg": "request", "request_id": "21ed1585ec44", "method": "POST", "path": "/predict", "status": 200, "latency_ms": 27.43, "client": "testclient"}


200 {'prediction': 0, 'label': 'no heart disease', 'confidence': 0.985, 'probability_disease': 0.015, 'risk_band': 'low', 'model_name': 'logistic_regression', 'model_version': '1.0.0'}
{"ts": "2026-10-08T11:03:32-0400", "level": "INFO", "logger": "heart.api", "msg": "prediction", "prediction": 1, "probability": 0.9471}


{"ts": "2026-10-08T11:03:32-0400", "level": "INFO", "logger": "heart.api", "msg": "request", "request_id": "a27bdf9c6157", "method": "POST", "path": "/predict", "status": 200, "latency_ms": 23.64, "client": "testclient"}


200 {'prediction': 1, 'label': 'heart disease', 'confidence': 0.9471, 'probability_disease': 0.9471, 'risk_band': 'high', 'model_name': 'logistic_regression', 'model_version': '1.0.0'}
{"ts": "2026-10-08T11:03:32-0400", "level": "INFO", "logger": "heart.api", "msg": "request", "request_id": "260e44fbe2c7", "method": "POST", "path": "/predict", "status": 422, "latency_ms": 1.88, "client": "testclient"}


422 Field required


Against the running container / Kubernetes service:
```bash
curl -X POST http://localhost:8000/predict -H "Content-Type: application/json" \
     -d @samples/patient_high_risk.json
```